# Practice Lab: Algorithmic Efficiency and Complexity Challenges

This lab provides hands-on problem-solving exercises to consolidate time complexity analysis, asymptotic ranking, loop counting, and empirical algorithm optimization.

## Lab Objectives

1. **Challenge 1**: Asymptotic Expression Ranking.
2. **Challenge 2**: Tracing Complex Loops and Step Formulations.
3. **Challenge 3**: Optimizing the Two-Sum Problem from $O(n^2)$ to $O(n)$.
4. **Challenge 4**: Prefix-Sum Optimization for Range Queries.
5. **Challenge 5**: Empirical Log-Log Slope Estimation for Empirical Degree.

## Challenge 1: Asymptotic Ranking

Order the following functions in non-decreasing order of asymptotic growth rate:

$$g_1(n) = 2^{10}, \quad g_2(n) = 2^{\log_2 n}, \quad g_3(n) = 4n\log_2 n + 2n, \quad g_4(n) = 4^{\log_2 n}, \quad g_5(n) = n^{1/\log_2 n}$$

We verify our ranking by evaluating at large $n$.

In [ ]:
import math

# Evaluated at large n = 1024
n = 1024
g1 = 2**10  # Constant: 1024
g2 = 2 ** math.log2(n)  # Linear: n = 1024
g3 = 4 * n * math.log2(n) + 2 * n  # n log n: 40,960 + 2048 = 43,008
g4 = 4 ** math.log2(n)  # Quadratic: n^2 = 1,048,576
g5 = n ** (1.0 / math.log2(n))  # Constant: 2.0

ranking = [
    ("g5: n^(1/log n)", g5),
    ("g1: 2^10", g1),
    ("g2: 2^(log n)", g2),
    ("g3: 4n log n", g3),
    ("g4: 4^(log n)", g4),
]
ranking_sorted = sorted(ranking, key=lambda item: item[1])
print("Asymptotic Growth Order (low to high):")
for rank, (name, val) in enumerate(ranking_sorted, 1):
    print(f"{rank}. {name} (val at n={n}: {val:,.1f})")

## Challenge 2: Tracing Nested Loop Iterations

Analyze the mystery function below and determine its exact execution count and Big-O complexity.

In [ ]:
import math


def mystery_algorithm(n: int) -> int:
    """Count operations executed across nested step loops."""
    steps = 0
    for _ in range(n):
        j = 1
        while j < n:
            steps += 1
            j *= 2
    return steps


for test_n in [4, 8, 16, 32]:
    actual = mystery_algorithm(test_n)
    expected = test_n * math.ceil(math.log2(test_n))
    print(f"n = {test_n:2d} | steps = {actual:3d} | expected n*ceil(log2 n) = {expected:3d}")
    assert actual == expected
print("Verified: mystery_algorithm executes Theta(n log n) steps.")

## Challenge 3: Two-Sum Optimization from $O(n^2)$ to $O(n)$

Given an unsorted list of integers and a target sum $T$, find two numbers that sum to $T$.

In [ ]:
import time


def two_sum_brute_force(arr: list[int], target: int) -> tuple[int, int] | None:
    """O(n^2) brute force search over all pairs."""
    n = len(arr)
    for i in range(n):
        for j in range(i + 1, n):
            if arr[i] + arr[j] == target:
                return i, j
    return None


def two_sum_hash_map(arr: list[int], target: int) -> tuple[int, int] | None:
    """O(n) hash table lookup."""
    seen: dict[int, int] = {}
    for idx, val in enumerate(arr):
        complement = target - val
        if complement in seen:
            return seen[complement], idx
        seen[val] = idx
    return None


# Benchmark with n = 5,000
test_list = list(range(1, 5001))
target_sum = 4999 + 5000

t0 = time.perf_counter()
p1 = two_sum_brute_force(test_list, target_sum)
t_brute = time.perf_counter() - t0

t0 = time.perf_counter()
p2 = two_sum_hash_map(test_list, target_sum)
t_hash = time.perf_counter() - t0

assert p1 == p2
print(f"Brute force O(n^2): {t_brute * 1000:.2f} ms")
print(f"Hash map    O(n)  : {t_hash * 1000:.4f} ms")
print(f"Speedup: {t_brute / t_hash:,.1f}x faster!")

## Challenge 4: Range Sum Queries with Prefix Sums

Given an array $A$ of size $n$, answer $q$ queries: compute $\sum_{k=L}^R A[k]$.

- **Naive**: loop from $L$ to $R$ on each query $\implies O(q \cdot n)$.
- **Prefix Sums**: precompute $P[i] = \sum_{k=0}^{i-1} A[k]$ in $O(n)$ time. Then each query is answered in $O(1)$ time via $P[R + 1] - P[L]$ $\implies O(n + q)$.

In [ ]:
class PrefixSumArray:
    def __init__(self, arr: list[int]):
        self.prefix = [0] * (len(arr) + 1)
        for idx, val in enumerate(arr):
            self.prefix[idx + 1] = self.prefix[idx] + val

    def query(self, left: int, right: int) -> int:
        """Return sum of arr[left..right] inclusive in O(1) time."""
        return self.prefix[right + 1] - self.prefix[left]


sample = [2, 4, 6, 8, 10, 12]
psa = PrefixSumArray(sample)
assert psa.query(1, 3) == 4 + 6 + 8
assert psa.query(0, 5) == sum(sample)
print("Prefix sum query correctness verified: O(1) per query!")

## Challenge 5: Empirical Log-Log Slope Estimation

Suppose an algorithm has execution time $T(n) = c \cdot n^k$. Taking logarithms:
$$\log T(n) = \log c + k \log n$$

The slope of $\log T(n)$ plotted against $\log n$ gives the empirical exponent $k$.

In [ ]:
import time

import numpy as np


def quadratic_task(n: int) -> int:
    total = 0
    for i in range(n):
        for j in range(n):
            total += i ^ j
    return total


sizes = [100, 200, 400, 800, 1600]
measured_times = []

for sz in sizes:
    t_start = time.perf_counter()
    quadratic_task(sz)
    measured_times.append(time.perf_counter() - t_start)

# Fit line on log-log values
log_sizes = np.log(sizes)
log_times = np.log(measured_times)
slope, intercept = np.polyfit(log_sizes, log_times, 1)

print(f"Empirical log-log slope: {slope:.2f}")
print("Expected slope for O(n^2): 2.00")
assert 1.7 <= slope <= 2.3  # verifies empirical quadratic growth
print("Empirical scaling matches theoretical O(n^2) prediction!")